# SciPy

A monthly atmospheric CO₂ record contains long-term change and a repeating seasonal pattern. Build a transparent numerical model, reserve later months for evaluation, and inspect residual structure. Small toolbox examples first establish the numerical ideas used in that investigation.

SciPy brings numerical algorithms for integration, interpolation, optimization, statistics, and signal processing.


## 1. Import the tools

The important SciPy functions here are:

- `curve_fit`: estimate the parameters of a function from observations.
- `savgol_filter`: create a smooth version of an ordered signal.
- `norm` and `pearsonr`: calculate statistical quantities.

Run the cells from top to bottom so variables are created in the expected order.

In [ ]:
import csv
import io
from pathlib import Path
from dataset_cache import get_dataset

import matplotlib.pyplot as plt
import numpy as np
from scipy import constants, integrate, interpolate, linalg, sparse, spatial
from scipy.optimize import curve_fit, least_squares, minimize, root_scalar
from scipy.signal import find_peaks, savgol_filter
from scipy.stats import describe, norm, pearsonr

OUTPUT_PLOT = Path.cwd() / "co2_seasonal_trend_notebook.png"

## SciPy fundamentals: the important toolbox

SciPy is a collection of numerical algorithms. The useful first question is not "Which function should I memorize?" but "What kind of mathematical problem do I have?" Once the problem is clear, the subpackage points us toward an algorithm.

| Problem we want to solve | SciPy area | Example function | What the result helps us understand |
| --- | --- | --- | --- |
| Accumulate a changing quantity | `scipy.integrate` | `quad`, `solve_ivp` | Area over an interval or a system's evolution over time |
| Estimate values between measurements | `scipy.interpolate` | `CubicSpline` | A continuous estimate between known points |
| Find a best value or solve an equation | `scipy.optimize` | `minimize`, `root_scalar`, `curve_fit` | A minimum, a root, or fitted model parameters |
| Describe variation or uncertainty | `scipy.stats` | `describe`, `norm`, `pearsonr` | Summaries, probabilities, or association measures |
| Clean or inspect an ordered signal | `scipy.signal` | `savgol_filter`, `find_peaks` | A smoother signal or candidate event locations |
| Solve equations with matrices | `scipy.linalg` | `solve`, `svd`, `eigh` | Solutions and structure in linear systems |
| Store or solve large mostly-empty matrices | `scipy.sparse` | `csr_matrix`, `diags`, `spsolve` | Lower memory use and sparse-system solutions |
| Measure geometry or find nearby points | `scipy.spatial` | `distance`, `KDTree` | Distances and efficient neighbor searches |
| Evaluate less common mathematical functions | `scipy.special` | `gamma`, `erf`, `expit` | Building blocks used in statistics and scientific models |

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Start from the question, then choose the tool.</strong><br>
If we know a function and want its accumulated area, we think integration. If we know observations and want a curve's parameters, we think curve fitting. The algorithm is useful only after we state what the data represents and what answer we need.
</div>

SciPy builds on NumPy, so array shape, units, missing values, and numerical scale still matter. A SciPy function is an algorithm, not automatically a complete scientific or machine-learning solution: we still define the question, check assumptions, prevent leakage, and interpret the result.

### Small examples of the core APIs

In [ ]:
# SciPy constants are named values with documented scientific units.
speed_of_light = constants.c
print(f"Speed of light: {speed_of_light:.0f} m/s")

### Integration: accumulate a quantity

Imagine a tank is filling at a rate that changes over time. Knowing the rate at a few individual moments is not enough; we want the total amount added across an interval. The area under the rate curve gives that accumulated total. `integrate.quad` estimates this area numerically.

| Part of the call | Example | Meaning |
| --- | --- | --- |
| Function | `lambda x: x**2` | The value being accumulated at each input `x` |
| Lower bound | `0` | Start of the interval |
| Upper bound | `1` | End of the interval |
| Returned `area` | about `0.333` | Estimated integral, here $\int_0^1 x^2\,dx = 1/3$ |
| Returned error estimate | near `0` | SciPy's estimate of numerical integration error, not measurement uncertainty |

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Interpret the units as well as the number.</strong><br>
If `x` is time in seconds and the function is a rate in liters per second, the integral is liters. The limits and units are part of the problem, not decoration around the function call.
</div>

Use `integrate.quad` for a one-dimensional definite integral. For a differential equation describing how a state changes over time, the related `solve_ivp` function integrates that changing state.

In [ ]:
area, integration_error = integrate.quad(lambda x: x**2, 0, 1)
print(f"Integral of x^2 from 0 to 1: {area:.3f}")
print(f"Estimated numerical error: {integration_error:.2e}")

### Interpolation: estimate between observations

Suppose a sensor recorded values at a few times, but a later calculation needs a value between two recorded times. Interpolation builds an estimated function from known points; it does not create a new measurement.

| Known input `x` | Measured output `y` | Role in the example |
| ---: | ---: | --- |
| 0.0 | 0.0 | First supplied point |
| 1.0 | 1.0 | Middle supplied point |
| 2.0 | 0.0 | Last supplied point |
| 1.5 | Not measured | Location where `spline(1.5)` estimates a value |

`CubicSpline(known_x, known_y)` constructs a smooth piecewise-cubic curve through the measured points. Calling `spline(1.5)` evaluates that constructed curve at the requested input.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Interpolation stays between known inputs; extrapolation goes beyond them.</strong><br>
A spline can return numbers outside the measured range, but those values are not supported by the observations in the same way. Use interpolation for suitable gaps and be cautious about interpreting extrapolated values as evidence.
</div>

In [ ]:
known_x = np.array([0.0, 1.0, 2.0])
known_y = np.array([0.0, 1.0, 0.0])
spline = interpolate.CubicSpline(known_x, known_y)
interpolated_value = spline(1.5)
print(f"Interpolated value at x=1.5: {interpolated_value:.3f}")

### Optimization: minimize, solve, and fit

Optimization functions all search for numerical answers, but they answer different questions. Choosing the right function starts with naming the unknown and the condition it must satisfy.

| Function | Question it answers | Example from the next cell | Result to inspect |
| --- | --- | --- | --- |
| `minimize` | Which input makes an objective as small as possible? | Minimize `(x - 3)²` | The minimizing input, near `x = 3` |
| `root_scalar` | Which scalar input makes a function equal zero? | Solve `x² - 2 = 0` | A root near `x = 1.414` within the bracket |
| `least_squares` | Which parameters make residuals collectively small? | Fit slope and intercept to observations | Parameter vector and residual behavior |
| `curve_fit` | Which parameters make a model curve fit observed data? | Fit the seasonal model later | Estimated parameters and covariance estimate |

For `root_scalar`, the symbols have direct meanings: `x` is the unknown input, `x² - 2` is the function whose zero we want, and `[0, 2]` is a bracket containing the positive root because the function changes sign between those endpoints.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>The output is numerical, so check whether it answers the stated question.</strong><br>
A successful optimizer reports an answer according to its objective and constraints. It does not prove that the objective represents the real problem or that the data support the answer.
</div>

Use `minimize` when we define an objective, `root_scalar` when we need an input where a scalar function equals zero, and `least_squares` when residuals are the natural objective. `curve_fit` specializes this idea for fitting a model to observations.

In [ ]:
minimum = minimize(lambda values: (values[0] - 3) ** 2, x0=[0.0])
root = root_scalar(lambda value: value**2 - 2, bracket=[0.0, 2.0]).root

linear_fit = least_squares(
    lambda parameters: parameters[0] * np.array([0.0, 1.0, 2.0])
    + parameters[1]
    - np.array([1.0, 3.0, 5.0]),
    x0=[0.0, 0.0],
)

print(f"Minimum near x={minimum.x[0]:.3f}")
print(f"Root near x={root:.3f}")
print(f"Least-squares parameters [slope, intercept]: {linear_fit.x}")

### Statistics: describe data and quantify uncertainty

Statistics helps us move from a handful of observations to statements about their center, spread, and relationships. The functions below answer different questions, so keep the measurement and assumptions in view.

| Function or value | Question it answers | Small interpretation |
| --- | --- | --- |
| `describe(sample)` | What are the count, mean, variance, minimum, and maximum? | For `[2, 4, 7, 9]`, the mean is `(2 + 4 + 7 + 9) / 4 = 5.5` |
| `norm.cdf(z)` | What fraction of a standard normal distribution lies at or below `z`? | A cumulative probability, between 0 and 1 |
| `norm.cdf(1.96) - norm.cdf(-1.96)` | What fraction lies between these two z-scores? | About 0.95 for a standard normal distribution |
| `pearsonr(x, y)` | How strongly do two numeric variables follow a linear pattern? | Returns a correlation coefficient and a p-value for a test of zero correlation |

In the `norm` example, $z$ is measured in standard deviations from the mean. The interval from -1.96 to 1.96 covers about 95% of a standard normal distribution; that statement only applies under the standard-normal model.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Association is not a cause-and-effect explanation.</strong><br>
A strong Pearson correlation describes a linear association in the measured data. It does not tell us that changing one variable will cause the other to change.
</div>

`describe` gives a compact summary. Distribution objects such as `norm` provide probabilities and quantiles. Correlation functions such as `pearsonr` measure linear association; they do not prove causation.

In [ ]:
sample = np.array([2.0, 4.0, 7.0, 9.0])
summary = describe(sample)
probability_inside_normal_range = norm.cdf(1.96) - norm.cdf(-1.96)

print(f"Mean: {summary.mean:.3f}")
print(f"Variance: {summary.variance:.3f}")
print(f"Standard-normal probability inside +/-1.96: {probability_inside_normal_range:.3f}")

### Signal processing: smooth and detect events

Signal functions expect ordered observations with a meaningful sampling order. `savgol_filter` can smooth while preserving local shape, and `find_peaks` can locate prominent local maxima. Filter choices should respect the sampling rate and should not leak future information into a training set.

In [ ]:
signal = np.array([0.0, 1.0, 0.0, 2.0, 0.0])
peaks, peak_properties = find_peaks(signal, prominence=0.5)
print(f"Detected peak indexes: {peaks}")
print(f"Peak prominences: {peak_properties['prominences']}")

### Linear algebra: solve systems safely

For a system $Ax=b$, use `linalg.solve(A, b)` instead of explicitly calculating an inverse. Other important tools include `svd` for decompositions and `eigh` for symmetric or Hermitian eigenvalue problems.

In [ ]:
matrix = np.array([[3.0, 1.0], [1.0, 2.0]])
right_side = np.array([9.0, 8.0])
matrix_solution = linalg.solve(matrix, right_side)
print(f"Solution to A*x=b: {matrix_solution}")

### Sparse matrices: work with mostly-zero data

A sparse matrix avoids storing all the zero entries in a large structure. `diags` is convenient for banded systems, while `csr_matrix` is a common general-purpose format. Sparse algorithms matter when the dense equivalent would waste memory.

In [ ]:
diagonal_matrix = sparse.diags([1.0, 2.0, 3.0])
print(f"Sparse matrix shape: {diagonal_matrix.shape}")
print(f"Stored non-zero values: {diagonal_matrix.nnz}")

### Spatial algorithms: compare points and search neighbors

`spatial.distance` measures distances between feature vectors. `KDTree` can make repeated nearest-neighbor queries efficient for many points. Be careful with units and feature scaling: distance is only meaningful when the coordinates are comparable.

In [ ]:
point_a = np.array([0.0, 0.0])
point_b = np.array([3.0, 4.0])
euclidean_distance = spatial.distance.euclidean(point_a, point_b)
print(f"Euclidean distance: {euclidean_distance:.3f}")

## 2. Inspect what the measurement record actually contains

Check dates, missing-value conventions and measurement units before fitting. Verify ordering and coverage. A historical record can be updated by its provider, so preserve the cached snapshot when comparing repeated analyses.

NOAA publishes comment lines followed by comma-separated rows. The `average` column is the monthly mean CO2 concentration in parts per million (ppm). Negative averages are missing-value markers, so we skip them.

In [ ]:
csv_text = get_dataset("co2_mm_mlo.csv").read_text(encoding="utf-8")

decimal_years = []
monthly_averages = []
for row in csv.reader(io.StringIO(csv_text)):
    if not row or row[0].startswith("#") or row[0] == "year":
        continue
    try:
        decimal_year = float(row[2])
        average = float(row[3])
    except (IndexError, ValueError):
        continue
    if average >= 0:
        decimal_years.append(decimal_year)
        monthly_averages.append(average)

decimal_years = np.asarray(decimal_years)
monthly_averages = np.asarray(monthly_averages)
print(f"{len(monthly_averages)} observations from {decimal_years[0]:.2f} to {decimal_years[-1]:.2f}")

## 3. Choose a model whose components have an interpretation

Identify which term represents long-term change and which represents seasonality. Predict the effect of changing each parameter. A good visual fit does not establish that the chosen trend will extrapolate indefinitely.

The CO2 measurements rise over the years, but they also move up and down within each year. A straight line can represent the long-term rise, but it cannot represent the repeating seasonal cycle. We combine a trend with sine and cosine terms to describe both patterns.

$$C(t)=a+b(t-t_0)+c_1\sin(2\pi t)+d_1\cos(2\pi t)+c_2\sin(4\pi t)+d_2\cos(4\pi t)$$

| Symbol in the equation | Python parameter | Meaning and units |
| --- | --- | --- |
| $C(t)$ | Function return value | Predicted CO2 concentration at time $t$, in ppm |
| $t$ | `decimal_year` | Time of an observation, measured in years |
| $t_0$ | `reference_year` | Start year used as the origin for the linear trend |
| $a$ | `intercept` | Baseline component of the trend at $t_0$, in ppm, before seasonal terms are added |
| $b$ | `slope` | Long-term change per year, in ppm/year |
| $c_1,d_1$ | `annual_sine`, `annual_cosine` | Coefficients that set the size and timing of the yearly cycle |
| $c_2,d_2$ | `half_year_sine`, `half_year_cosine` | Coefficients that let the yearly shape be less like a simple sine wave |

For example, if $b=2$, then the linear trend term adds 2 ppm for each year after $t_0$. The sine and cosine terms move the prediction above or below that trend as the year changes. Using both sine and cosine lets the fitted cycle peak at a phase supported by the data instead of forcing January to be the peak.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Keep the time origin consistent between training and prediction.</strong><br>
The slope uses $t-t_0$. When we evaluate the held-out future, we reuse the training start year as $t_0$; changing it would shift the trend's baseline and make the fitted intercept mean something different.
</div>

The reference year matters: during prediction, we must keep the training period's time origin.

In [ ]:
def seasonal_trend_from_anchor(
    decimal_year,
    reference_year,
    intercept,
    slope,
    annual_sine,
    annual_cosine,
    half_year_sine,
    half_year_cosine,
):
    years_from_start = decimal_year - reference_year
    annual_angle = 2 * np.pi * decimal_year
    return (
        intercept
        + slope * years_from_start
        + annual_sine * np.sin(annual_angle)
        + annual_cosine * np.cos(annual_angle)
        + half_year_sine * np.sin(2 * annual_angle)
        + half_year_cosine * np.cos(2 * annual_angle)
    )


def seasonal_trend(decimal_year, intercept, slope, annual_sine, annual_cosine, half_year_sine, half_year_cosine):
    return seasonal_trend_from_anchor(
        decimal_year, decimal_year[0], intercept, slope, annual_sine,
        annual_cosine, half_year_sine, half_year_cosine
    )

## 4. Hold back the future rather than random months

Locate the chronological boundary before fitting. Explain why a random split would give the model access to later observations during an earlier-period evaluation. Keep the same holdout when comparing prespecified alternatives.

The last 20% of a time series represents the future, so it becomes the test set. We fit only on earlier observations. A random split would allow observations from the future to influence training and would answer a different question.

In [ ]:
split_index = int(len(decimal_years) * 0.8)
training_years = decimal_years[:split_index]
training_averages = monthly_averages[:split_index]
test_years = decimal_years[split_index:]
test_averages = monthly_averages[split_index:]

print(f"Training months: {len(training_years)}")
print(f"Test months: {len(test_years)}")
print(f"Test begins at decimal year {test_years[0]:.2f}")

## 5. Fit with `scipy.optimize.curve_fit`

`curve_fit` searches for parameters that make the model's predictions close to the observations. The covariance matrix gives an approximate measure of parameter uncertainty under the fitting assumptions.

In [ ]:
initial_guess = [training_averages[0], 2.0, 2.0, 2.0, 0.2, 0.2]
training_parameters, training_covariance = curve_fit(
    seasonal_trend,
    training_years,
    training_averages,
    p0=initial_guess,
    maxfev=20_000,
)

test_predictions = seasonal_trend_from_anchor(
    test_years, training_years[0], *training_parameters
)
print("Fitted parameters:")
for name, value in zip(
    ["intercept", "slope", "annual_sine", "annual_cosine", "half_year_sine", "half_year_cosine"],
    training_parameters,
):
    print(f"  {name}: {value:.4f}")

## 6. Check where the forecast fails, not only its average error

Compare predictions with observed held-out months in the original units. Look for systematic seasonal or trend errors. A small average error can conceal time-dependent bias, and a full-record refit is not an independent new test.

MAE is the average absolute error. RMSE also averages errors but penalizes large errors more strongly. These are evaluation measures, not special SciPy models.

In [ ]:
test_mae = np.mean(np.abs(test_averages - test_predictions))
test_rmse = np.sqrt(np.mean((test_averages - test_predictions) ** 2))
print(f"Test MAE: {test_mae:.3f} ppm")
print(f"Test RMSE: {test_rmse:.3f} ppm")

## 7. Fit the full record and inspect uncertainty

The holdout metrics above remain the honest future-style evaluation. We now fit all available observations only to describe the full record and make the final plot. The slope interval is approximate and depends on the model and residual assumptions.

In [ ]:
full_parameters, full_covariance = curve_fit(
    seasonal_trend, decimal_years, monthly_averages, p0=initial_guess, maxfev=20_000
)
slope = full_parameters[1]
slope_standard_error = np.sqrt(full_covariance[1, 1])
slope_interval = norm.ppf(0.975) * slope_standard_error
full_predictions = seasonal_trend(decimal_years, *full_parameters)
residuals = monthly_averages - full_predictions
residual_correlation = pearsonr(decimal_years, residuals)

print(f"Full-record slope: {slope:.3f} +/- {slope_interval:.3f} ppm/year")
print(f"Residual/time correlation: {residual_correlation.statistic:.3f}")
print(f"Residual/time p-value: {residual_correlation.pvalue:.3g}")

## 8. Smooth and visualize

The Savitzky-Golay filter is used for a readable smooth line, not as a replacement for measured data. The vertical line marks where the held-out test period begins.

In [ ]:
window_length = min(25, len(monthly_averages) if len(monthly_averages) % 2 else len(monthly_averages) - 1)
smoothed_averages = savgol_filter(monthly_averages, window_length=window_length, polyorder=2)

plt.figure(figsize=(11, 6))
plt.scatter(decimal_years, monthly_averages, s=5, alpha=0.35, label="NOAA monthly mean")
plt.plot(decimal_years, smoothed_averages, linewidth=2, label="Savitzky-Golay smooth")
plt.plot(decimal_years, full_predictions, linewidth=2, label="Fitted trend + seasonality")
plt.axvline(test_years[0], color="black", linestyle="--", label="Test starts")
plt.xlabel("Year")
plt.ylabel("CO2 (ppm)")
plt.title("Mauna Loa atmospheric CO2: SciPy seasonal trend")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=150)
plt.show()
print(f"Plot written to: {OUTPUT_PLOT}")

## Explain what the evidence supports

Describe the model’s trend and seasonal components, its held-out error and one pattern left unexplained. State the snapshot and time interval to which that assessment applies.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: fit a decay calibration despite corrupted measurements

A lab-like sensor export contains duplicate timestamps, missing values and several large training-period spikes. Compare ordinary and robust nonlinear least squares on a positive exponential-decay model, using later observations only for evaluation. The data is generated with known parameters so the exercise can separate implementation checks from statistical claims.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through why the residual scale matters

Ordinary least squares squares every residual: an error of 10 contributes 100 while an error of 1 contributes 1. A few spikes can dominate the fit. A soft-L1 loss reduces the influence of large residuals, but its scale defines what counts as large. Here a known teaching noise scale of 0.15 fixes that choice before holdout evaluation. Robust loss is not permission to delete inconvenient observations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from scipy.optimize import least_squares
p2_rng=np.random.default_rng(22)
p2_time=np.linspace(0,8,120)
p2_truth=4.0*np.exp(-0.45*p2_time)+0.7
p2_observed=p2_truth+p2_rng.normal(0,.15,120)
p2_observed[[18,44,70]] += [5,-4,6];p2_observed[[3,31]]=np.nan
p2_raw=pd.DataFrame({'time':p2_time,'reading':p2_observed})
p2_raw=pd.concat([p2_raw,p2_raw.iloc[20:25]],ignore_index=True)
p2_clean=p2_raw.drop_duplicates().dropna().sort_values('time')
assert len(p2_raw)==125 and len(p2_clean)==118
p2_train=p2_clean[p2_clean.time<p2_time[90]];p2_test=p2_clean[p2_clean.time>=p2_time[90]]
def p2_predict(params,t):return params[0]*np.exp(-params[1]*t)+params[2]
def p2_residual(params):return p2_predict(params,p2_train.time.to_numpy())-p2_train.reading.to_numpy()
p2_fits={p2_loss:least_squares(p2_residual,[3,.3,.5],bounds=([0,0,0],[10,3,3]),loss=p2_loss,f_scale=.15,max_nfev=500)
         for p2_loss in ['linear','soft_l1']}
assert all(f.success and np.isfinite(f.x).all() for f in p2_fits.values())

### Work through the held-out comparison and solver budget

Compare RMSE on the same later measurements, parameter estimates and solver evaluations. `max_nfev=500` bounds a solver counter, not total runtime across all residual/Jacobian work. Print termination messages rather than accepting any returned vector as a converged fit.

The model is constrained to positive amplitude, decay and background, reflecting the synthetic generating mechanism. For a real sensor, justify those constraints from domain knowledge and inspect residual patterns that suggest the chosen function is wrong.

In [ ]:
p2_rows=[]
for p2_name,p2_fit in p2_fits.items():
    p2_prediction=p2_predict(p2_fit.x,p2_test.time.to_numpy())
    p2_rows.append({'loss':p2_name,'amplitude':p2_fit.x[0],'decay':p2_fit.x[1],'background':p2_fit.x[2],
                   'holdout_rmse':np.sqrt(np.mean((p2_prediction-p2_test.reading.to_numpy())**2)),
                   'nfev':p2_fit.nfev,'termination':p2_fit.message})
p2_report=pd.DataFrame(p2_rows);display(p2_report)
import matplotlib.pyplot as plt
p2_fig,p2_ax=plt.subplots(figsize=(8,4));p2_ax.scatter(p2_train.time,p2_train.reading,s=12,label='Training observations')
p2_ax.scatter(p2_test.time,p2_test.reading,s=12,label='Later observations')
for p2_name,p2_fit in p2_fits.items():p2_ax.plot(p2_time,p2_predict(p2_fit.x,p2_time),label=p2_name)
p2_ax.axvline(p2_time[90],color='gray',linestyle='--');p2_ax.set(xlabel='Time',ylabel='Sensor reading');p2_ax.legend()
plt.tight_layout();plt.savefig(p2_out/'scipy_robust_calibration.png');plt.show()
p2_report.to_csv(p2_out/'scipy_calibration_comparison.csv',index=False)

### Explain what a robust fit establishes

The robust fit can resist isolated spikes when the majority of measurements follow the model. It cannot rescue systematically biased data or guarantee valid extrapolation far beyond observed times. Duplicate removal was exact-row removal, not averaging conflicting measurements at the same timestamp.

**Try:** double every reading after time 6 to represent a sensor change. **Worked reasoning:** robust training on earlier data cannot anticipate a new measurement regime. A poor later fit could indicate drift rather than an optimizer failure. Keep this diagnostic distinct from changing loss parameters until the holdout score improves.